In [43]:
import sys
from PyQt6 import QtCore, QtGui, QtWidgets, uic

class AddWindow(QtWidgets.QMainWindow):
  def __init__(self, parent=None):
    super().__init__()
    self.parent = parent
    uic.loadUi("AddUI.ui", self)

    # Define UI objects
    self.idLineEdit = self.findChild(QtWidgets.QLineEdit, "idLineEdit")
    self.fullnameLineEdit = self.findChild(QtWidgets.QLineEdit, "fullnameLineEdit")
    self.phoneLineEdit = self.findChild(QtWidgets.QLineEdit, "phoneLineEdit")
    self.addButton = self.findChild(QtWidgets.QPushButton, "addButton")

    # Button events
    self.addButton.clicked.connect(self.add_data)
    
  # Behaviors
  def add_data(self):
    id = self.idLineEdit.text()
    fullname = self.fullnameLineEdit.text()
    phone = self.phoneLineEdit.text()
    # print (f"{fullname} ({phone})")
    if not fullname or not phone:
      QtWidgets.QMessageBox.warning(self, "Warning!", "Please complete all fields")
      return
    
    self.parent.addUser(id, fullname, phone)
    self.close()

# Edit Window
class EditWindow(QtWidgets.QMainWindow):
  def __init__(self, parent=None, editedUserId=None, editedUserData=None):
    super().__init__()
    self.parent = parent
    self.editedUserId = editedUserId
    self.editedUserData = editedUserData
    uic.loadUi("EditUI.ui", self)

    # Define UI objects
    self.idLineEdit = self.findChild(QtWidgets.QLineEdit, "idLineEdit")
    self.fullnameLineEdit = self.findChild(QtWidgets.QLineEdit, "fullnameLineEdit")
    self.phoneLineEdit = self.findChild(QtWidgets.QLineEdit, "phoneLineEdit")
    self.updateButton = self.findChild(QtWidgets.QPushButton, "updateButton")

    # Button events
    self.updateButton.clicked.connect(self.edit_data)

    # Init edited value
    self.idLineEdit.setText(self.editedUserId)
    self.fullnameLineEdit.setText(self.editedUserData["fullname"])
    self.phoneLineEdit.setText(self.editedUserData["phone"])
    
  # Behaviors
  def edit_data(self):
    id = self.idLineEdit.text()
    fullname = self.fullnameLineEdit.text()
    phone = self.phoneLineEdit.text()
    if not fullname or not phone:
      QtWidgets.QMessageBox.warning(self, "Warning!", "Please complete all fields")
      return
    
    self.parent.editUser(id, fullname, phone)
    self.close()

In [44]:
class MainWindow(QtWidgets.QMainWindow):
  def __init__(self):
    super().__init__()
    uic.loadUi("MainUI.ui", self)

    # Window settings
    self.setWindowFlag(
      QtCore.Qt.WindowType.WindowCloseButtonHint
    )
    # self.setWindowIcon(QtGui.QIcon("Path.png"))

    # App variables
    self.users = {}
    self.editedUsersKey = None

    # Define UI object
    self.list = self.findChild(QtWidgets.QListWidget, "listWidget")
    self.addButton = self.findChild(QtWidgets.QPushButton, "addButton")
    self.editButton = self.findChild(QtWidgets.QPushButton, "editButton")
    self.deleteButton = self.findChild(QtWidgets.QPushButton, "deleteButton")
    self.showInfoButton = self.findChild(QtWidgets.QPushButton, "showInfoButton")

    # Button events
    self.addButton.clicked.connect(self.handleAddButton)
    self.editButton.clicked.connect(self.handleEditButton)
    self.deleteButton.clicked.connect(self.handleDeleteButton)
    self.showInfoButton.clicked.connect(self.handleShowInfoButton)

  # Behaviors
  def handleAddButton(self):
    # Open sub-windows (add windows)
    self.addwindows = AddWindow(self)
    self.addwindows.show()

  def handleEditButton(self):
    # Open sub-windows (edit windows)
    currentItem = self.list.currentItem()
    if not currentItem:
      QtWidgets.QMessageBox.warning(self, "Warning!", "No user selected.")
      return
    
    editedUserId = currentItem.data(QtCore.Qt.ItemDataRole.UserRole)
    editedUserData = self.users[editedUserId]

    self.editWindows = EditWindow(self, editedUserId, editedUserData)
    self.editWindows.show()

  def handleDeleteButton(self):
    deletedItem = self.list.currentItem()
    if not deletedItem:
      QtWidgets.QMessageBox.warning(self, "Warning!", "No user selected.")
      return

    messageBox = QtWidgets.QMessageBox()
    messageBox.setWindowTitle("Delete confirmation")
    messageBox.setText("Are you sure?")
    messageBox.setStandardButtons(messageBox.StandardButton.Ok | messageBox.StandardButton.Cancel)
    messageBox.setIcon(messageBox.Icon.Warning)

    confirm = messageBox.exec()

    if (confirm == messageBox.StandardButton.Ok):
      deletedId = deletedItem.data(QtCore.Qt.ItemDataRole.UserRole)
      deletedIndex = self.list.row(deletedItem)
      print(f"Deleted Id: {deletedId}: {self.users[deletedId]}")
      self.list.takeItem(deletedIndex)
      self.users[deletedId] = None
      print(f"Current users: {self.users}")
  
  def handleShowInfoButton(self):
    messageBox = QtWidgets.QMessageBox()
    currentFocusedList = self.list.currentRow()
    key = self.list.currentItem().text()
    print(f"Debug: {currentFocusedList}, {key}, {self.users[key]}")

    messageBox.setWindowTitle("Debug data")
    messageBox.setStandardButtons(messageBox.StandardButton.Ok | messageBox.StandardButton.Cancel)

    messageBoxBody = f"Key: {key} \nValue: {self.users[key]}\n"

    messageBox.setText(messageBoxBody)
    messageBox.setIcon(messageBox.Icon.Information)
    result = messageBox.exec()

    if (result == messageBox.StandardButton.Ok):
      print("OK")
    
  def addUser(self, id, fullname, phone):
    # Validate duplicated ID
    if id in self.users:
      QtWidgets.QMessageBox.warning(self, "Warning!", f"{id} already exists.")
      return

    self.users[id] = {
      "fullname": fullname,
      "phone": phone
    }

    newItem = QtWidgets.QListWidgetItem(f"{fullname} ({phone})")
    newItem.setData(QtCore.Qt.ItemDataRole.UserRole, id)
    self.list.addItem(newItem)
    print("Added new user")
    print(self.users)

  def editUser(self, id, fullname, phone):
    self.users[id]["fullname"] = fullname
    self.users[id]["phone"] = phone

    # Update UI
    editedItem = self.list.currentItem()
    if not editedItem:
      return

    editedItem.setText(f"{fullname} ({phone})")
    
    print(f"Edited Id: {id}: ({self.users[id]})")
    print(f"Current users: {self.users}")

In [45]:
app = QtCore.QCoreApplication.instance()
if app is None : app = QtWidgets.QApplication([])
window = MainWindow()
window.show()
app.exec()

Added new user
{'1': {'fullname': '1', 'phone': '1'}}
Added new user
{'1': {'fullname': '1', 'phone': '1'}, '2': {'fullname': '1', 'phone': '1'}}
Edited Id: 2: ({'fullname': 'Poorin', 'phone': '094-204-1773'})
Current users: {'1': {'fullname': '1', 'phone': '1'}, '2': {'fullname': 'Poorin', 'phone': '094-204-1773'}}
Deleted Id: 1: {'fullname': '1', 'phone': '1'}
Current users: {'1': None, '2': {'fullname': 'Poorin', 'phone': '094-204-1773'}}


0